# 00 - Diagnostico, alcance e ingesta reproducible

Este notebook es el punto de entrada del proyecto. Su objetivo es construir una version canonica y reproducible del dataset ECtHR Task B para que todo el analisis posterior use exactamente los mismos casos, textos, etiquetas y particiones.

El problema se formula como un sistema de apoyo al analisis inicial de casos ante el Tribunal Europeo de Derechos Humanos. Dado el texto de los hechos, el sistema debe ayudar a tres tareas practicas: sugerir articulos potencialmente implicados, recuperar precedentes similares y producir explicaciones auxiliares que permitan auditar por que el modelo ha activado ciertas etiquetas.

La idea de principio a fin es:

```text
caso juridico en texto
        ↓
normalizacion y union de parrafos
        ↓
tabla de casos + tabla de etiquetas
        ↓
representacion numerica en los notebooks siguientes
        ↓
modelos, retrieval, explicabilidad y drift
```

En este primer cuaderno todavia no se entrena ningun modelo. Aqui se fijan las bases del experimento: que entra, como se guarda y que significa cada tabla.


In [7]:
from pathlib import Path
import json, sqlite3, re, random, math, warnings, sys, subprocess, importlib.util
from datetime import datetime, timezone
from uuid import uuid4
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd().resolve()
DATA = ROOT / 'data'
RAW = DATA / 'raw'
INTERIM = DATA / 'interim'
PROCESSED = DATA / 'processed'
ARTIFACTS = ROOT / 'artifacts'
FIGURES = ARTIFACTS / 'figures'
METRICS = ARTIFACTS / 'metrics'
MODELS = ARTIFACTS / 'models'
INDICES = ARTIFACTS / 'indices'
REPORTS = ARTIFACTS / 'reports'
DB = INTERIM / 'metadata.db'
SCHEMA_PATH = ROOT / 'schema.sql'
for d in [RAW, INTERIM, PROCESSED, FIGURES, METRICS, MODELS, INDICES, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package = pip_name or import_name
        print(f'Instalando {package} en este kernel...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

def dump_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

print('ROOT =', ROOT)


ROOT = /Users/jordiblascolozano/Documents/proyecto aprendizaje avanzado


## Ejemplo minimo: de parrafos juridicos a una fila tabular

Un caso del TEDH puede venir segmentado en varios parrafos o fragmentos. Para el modelo necesitamos una unidad documental completa: `text_full`. Por eso se unen los fragmentos preservando el orden y se calculan metadatos basicos.

El ejemplo siguiente usa un caso ficticio pequeno para mostrar exactamente que se guarda: identificador, texto unido, numero de parrafos, numero aproximado de tokens y etiquetas asociadas.


In [8]:
toy_case = {
    'case_id': 'toy_case_001',
    'split': 'train',
    'year': 2018,
    'paragraphs': [
        'The applicant complained about unlawful detention.',
        'He also alleged that the domestic proceedings were not fair.'
    ],
    'article_ids': ['5', '6']
}

toy_text_full = ' '.join(toy_case['paragraphs'])
toy_cases = pd.DataFrame([{
    'case_id': toy_case['case_id'],
    'split': toy_case['split'],
    'year': toy_case['year'],
    'text_full': toy_text_full,
    'n_paragraphs': len(toy_case['paragraphs']),
    'n_tokens': len(toy_text_full.split()),
}])
toy_case_labels = pd.DataFrame([
    {'case_id': toy_case['case_id'], 'article_id': art, 'value': 1}
    for art in toy_case['article_ids']
])

print('Tabla cases:')
display(toy_cases)
print('Tabla case_labels:')
display(toy_case_labels)


Tabla cases:


,case_id,split,year,text_full,n_paragraphs,n_tokens
0,toy_case_001,train,2018,The applicant complained about unlawful detent...,2,16


Tabla case_labels:


,case_id,article_id,value
0,toy_case_001,5,1
1,toy_case_001,6,1


## Esquema de datos

Aunque ECtHR Task B es un dataset textual, lo organizamos como un conjunto de tablas auditables. Esta estructura permite separar cuatro conceptos que no deben mezclarse:

- `cases`: una fila por caso, con `case_id`, `split`, `year`, `text_full`, `n_paragraphs` y `n_tokens`.
- `case_paragraphs`: los fragmentos originales del caso, utiles para inspeccion y futuras extensiones por parrafo.
- `articles`: el catalogo de etiquetas posibles.
- `case_labels`: la relacion multietiqueta entre casos y articulos.
- `predictions`, `experiment_runs` y `explanations`: resultados generados por los notebooks posteriores.

Separar datos, etiquetas y predicciones ayuda a evitar errores metodologicos. Por ejemplo, un caso de `test` puede tener etiquetas verdaderas para evaluar, pero esas etiquetas no deben usarse para entrenar ni para ajustar umbrales.


In [ ]:
def get_conn():
    conn = sqlite3.connect(DB)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_conn()
    conn.executescript(SCHEMA_PATH.read_text(encoding='utf-8'))
    conn.commit()
    return conn


## Ejemplo minimo: matriz multietiqueta `Y`

Como un caso puede activar varios articulos, la salida no es una unica clase. Se construye una matriz binaria `Y` con una fila por caso y una columna por articulo. Un `1` significa que el articulo aparece asociado al caso; un `0` significa que no.


In [10]:
toy_labels_long = pd.DataFrame([
    {'case_id': 'case_A', 'article_id': '5', 'value': 1},
    {'case_id': 'case_A', 'article_id': '6', 'value': 1},
    {'case_id': 'case_B', 'article_id': '8', 'value': 1},
    {'case_id': 'case_C', 'article_id': '6', 'value': 1},
])

toy_Y = toy_labels_long.pivot_table(
    index='case_id', columns='article_id', values='value', fill_value=0, aggfunc='max'
).astype(int)

display(toy_Y)
print('Y.shape =', toy_Y.shape, '= numero de casos x numero de articulos')


article_id,5,6,8
case_id,,,
case_A,1,1,0
case_B,0,0,1
case_C,0,1,0


Y.shape = (3, 3) = numero de casos x numero de articulos


## Funciones de normalizacion

Las funciones siguientes convierten entradas heterogeneas en una representacion estable. La normalizacion hace cuatro tareas:

1. Localiza el campo textual aunque el dataset lo nombre de forma distinta.
2. Une segmentos o parrafos en `text_full` sin cambiar el orden.
3. Extrae metadatos simples como longitud y numero de parrafos.
4. Mantiene un identificador estable para poder conectar casos, etiquetas, predicciones y explicaciones.

No se elimina informacion juridica de forma agresiva. En textos legales, palabras aparentemente repetidas o formulas estandar pueden ser senales utiles para ciertos articulos; por eso la limpieza se mantiene conservadora.


In [11]:
TEXT_KEYS = ('text', 'facts', 'fact', 'document', 'content')
ID_KEYS = ('case_id', 'id', 'doc_id')
YEAR_KEYS = ('year', 'decision_year', 'judgment_year', 'date', 'decision_date', 'judgment_date')

def extract_text_segments(example):
    text_value = ''
    for key in TEXT_KEYS:
        if key in example and example[key] is not None:
            text_value = example[key]
            break
    if isinstance(text_value, str):
        cleaned = text_value.strip()
        return [cleaned] if cleaned else []
    if isinstance(text_value, list):
        return [str(x).strip() for x in text_value if isinstance(x, str) and str(x).strip()]
    return []

def parse_year(value):
    if isinstance(value, bool):
        return None
    if isinstance(value, int) and 1900 <= value <= 2100:
        return value
    if isinstance(value, float) and value.is_integer() and 1900 <= int(value) <= 2100:
        return int(value)
    if isinstance(value, str):
        m = re.search(r'(19|20)\d{2}', value)
        if m:
            return int(m.group(0))
    return None

def extract_year(example):
    for key in YEAR_KEYS:
        if key in example and example[key] is not None:
            y = parse_year(example[key])
            if y is not None:
                return y
    return None

def extract_label_ids(raw_labels, label_count=None):
    if isinstance(raw_labels, (int, float, bool)):
        v = int(raw_labels)
        return [v] if v >= 0 else []
    if not isinstance(raw_labels, (list, tuple)) or len(raw_labels) == 0:
        return []
    values = [int(v) for v in raw_labels if isinstance(v, (int, float, bool))]
    if label_count is not None and len(values) == label_count and set(values).issubset({0, 1}):
        return [i for i, v in enumerate(values) if v == 1]
    return sorted({v for v in values if v >= 0})

def build_case_id(example, task, split, row_idx):
    for key in ID_KEYS:
        if key in example and example[key] is not None:
            safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(example[key])).strip('_')
            if safe:
                return f'{task}_{split}_{safe}'
    return f'{task}_{split}_{row_idx:06d}'

def build_case_record(example, task, split, row_idx):
    case_id = build_case_id(example, task, split, row_idx)
    segments = extract_text_segments(example)
    text_full = '\n\n'.join(segments)
    n_tokens = len(re.findall(r'\w+', text_full, flags=re.UNICODE))
    return {
        'case_id': case_id,
        'task': task,
        'split': split,
        'year': extract_year(example),
        'text_full': text_full,
        'n_paragraphs': len(segments),
        'n_tokens': n_tokens,
    }, segments


## Carga de LexGLUE ECtHR Task B

La tarea B de ECtHR es una clasificacion multietiqueta: un mismo caso puede estar asociado a varios articulos. En terminos de ML, cada caso se transforma en un vector binario `y` donde cada posicion indica si un articulo esta presente o no.

El notebook usa Hugging Face Datasets con cache local en `data/raw/hf_cache`. Para el informe y la defensa no se debe ejecutar con muestras diminutas: las metricas validas salen de la ingesta completa. Si se limita el numero de filas solo sirve como prueba tecnica, no como resultado academico.


In [12]:
ensure_package('datasets')
from datasets import load_dataset

TASK = 'ecthr_task_b'
DATASET_NAME = 'lex_glue'
DATASET_SUBSET = 'ecthr_b'
CACHE_DIR = RAW / 'hf_cache'

dataset = load_dataset(DATASET_NAME, DATASET_SUBSET, cache_dir=str(CACHE_DIR))
print(dataset)


Instalando datasets en este kernel...



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip3.11 install --upgrade pip


Generating train split:   0%|          | 0/9000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 1000
    })
})


In [13]:
def extract_label_names(dataset_dict):
    split_name = 'train' if 'train' in dataset_dict else list(dataset_dict.keys())[0]
    features = getattr(dataset_dict[split_name], 'features', None)
    if not features or 'labels' not in features:
        return []
    label_feature = features['labels']
    if hasattr(label_feature, 'feature') and hasattr(label_feature.feature, 'names'):
        return [str(x) for x in label_feature.feature.names]
    if hasattr(label_feature, 'names'):
        return [str(x) for x in label_feature.names]
    return []

label_names = extract_label_names(dataset)
label_count = len(label_names) if label_names else None
print('Labels:', label_names)


Labels: ['2', '3', '5', '6', '8', '9', '10', '11', '14', 'P1-1']


## Ingesta completa en SQLite

Esta celda materializa la version canonica del dataset usada por los demas notebooks. El resultado principal es `data/interim/metadata.db`, junto con un JSON de estado en `artifacts/reports/ingestion_status.json`.

El objetivo de guardar SQLite no es introducir una dependencia externa, sino fijar una frontera reproducible: todos los notebooks posteriores leen la misma tabla de casos y etiquetas. Asi se evita que cada experimento vuelva a interpretar el dataset de forma distinta.


In [14]:
conn = init_db()
case_rows, paragraph_rows, label_rows = [], [], []
discovered_labels = set()
split_counts = {}

for split_name in dataset.keys():
    split_counts[split_name] = 0
    for row_idx, example in enumerate(dataset[split_name]):
        case_record, segments = build_case_record(example, task=TASK, split=split_name, row_idx=row_idx)
        case_rows.append(case_record)
        for pidx, paragraph in enumerate(segments):
            paragraph_rows.append({'case_id': case_record['case_id'], 'paragraph_idx': pidx, 'paragraph_text': paragraph})
        labels = extract_label_ids(example.get('labels'), label_count=label_count)
        for label_id in labels:
            discovered_labels.add(label_id)
            label_rows.append({'case_id': case_record['case_id'], 'article_id': str(label_id), 'value': 1})
        split_counts[split_name] += 1

if label_names:
    article_rows = [{'article_id': str(i), 'article_code': name, 'description': ''} for i, name in enumerate(label_names)]
else:
    article_rows = [{'article_id': str(i), 'article_code': f'article_{i}', 'description': ''} for i in sorted(discovered_labels)]

with conn:
    conn.executemany('INSERT OR REPLACE INTO cases VALUES (:case_id, :task, :split, :year, :text_full, :n_paragraphs, :n_tokens)', case_rows)
    conn.executemany('INSERT OR REPLACE INTO case_paragraphs VALUES (:case_id, :paragraph_idx, :paragraph_text)', paragraph_rows)
    conn.executemany('INSERT OR REPLACE INTO articles VALUES (:article_id, :article_code, :description)', article_rows)
    conn.executemany('INSERT OR REPLACE INTO case_labels VALUES (:case_id, :article_id, :value)', label_rows)
    counts = {t: conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in ['cases','case_paragraphs','articles','case_labels','experiment_runs','predictions','explanations']}
conn.close()

report = {
    'stage': 'ingestion_notebook',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'dataset': {'name': DATASET_NAME, 'subset': DATASET_SUBSET, 'cache_dir': str(CACHE_DIR)},
    'task': TASK,
    'limit_per_split': None,
    'split_counts': split_counts,
    'inserted': {'articles': len(article_rows), 'cases': len(case_rows), 'case_paragraphs': len(paragraph_rows), 'case_labels': len(label_rows)},
    'table_counts': counts,
    'sqlite_path': str(DB),
}
dump_json(report, REPORTS / 'ingestion_status.json')
print(report)
assert counts['cases'] == 11000, 'La ingesta debe contener 11000 casos.'


{'stage': 'ingestion_notebook', 'timestamp_utc': '2026-04-28T11:45:57.015256+00:00', 'dataset': {'name': 'lex_glue', 'subset': 'ecthr_b', 'cache_dir': '/Users/jordiblascolozano/Documents/proyecto aprendizaje avanzado/data/raw/hf_cache'}, 'task': 'ecthr_task_b', 'limit_per_split': None, 'split_counts': {'train': 9000, 'test': 1000, 'validation': 1000}, 'inserted': {'articles': 10, 'cases': 11000, 'case_paragraphs': 262580, 'case_labels': 15991}, 'table_counts': {'cases': 11000, 'case_paragraphs': 262580, 'articles': 10, 'case_labels': 15991, 'experiment_runs': 0, 'predictions': 0, 'explanations': 0}, 'sqlite_path': '/Users/jordiblascolozano/Documents/proyecto aprendizaje avanzado/data/interim/metadata.db'}
